In [ ]:
from pathlib import Path
import json
import math
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown, clear_output


import ipywidgets as widgets

from scipy.signal import spectrogram


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
FOLDER_LEVELS_UP_BEFORE_PREPROCESSING = 2
PREPROCESSING_FOLDER_NAME = "data_preprocessing"
RAW_FEATURES_DIR_NAME = "Raw_Features"
NORMALIZED_DIR_NAME = "Normalized"

PREPROCESSING_BASE_DIR = Path.cwd()
for _ in range(FOLDER_LEVELS_UP_BEFORE_PREPROCESSING):
    PREPROCESSING_BASE_DIR = PREPROCESSING_BASE_DIR.parent

PREPROCESSING_ROOT = PREPROCESSING_BASE_DIR / PREPROCESSING_FOLDER_NAME
WIBRACJE_FOLDER = Path.cwd().parent.parent

DEFAULT_SEARCH_ROOTS = [
    PREPROCESSING_ROOT,
    WIBRACJE_FOLDER / "Datasety" / "Wibracje wyważanie" / "processed",
    WIBRACJE_FOLDER / "Datasety" / "Machine_Fault_Data" / "imbalance",
]

FILE_PATH = ""

print("PREPROCESSING_ROOT:", PREPROCESSING_ROOT)
print("Default search roots:")
for root in DEFAULT_SEARCH_ROOTS:
    print(" -", root)


PREPROCESSING_ROOT: c:\Users\szymo\Desktop\Wibracje\data_preprocessing
Default search roots:
 - c:\Users\szymo\Desktop\Wibracje\data_preprocessing
 - c:\Users\szymo\Desktop\Wibracje\Datasety\Wibracje wyważanie\processed
 - c:\Users\szymo\Desktop\Wibracje\Datasety\Machine_Fault_Data\imbalance


In [ ]:
# =============================================================================
# GENERAL HELPERS
# =============================================================================
def natural_key(value):
    text = str(Path(value).name)
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", text)]


def as_path(path):
    if isinstance(path, Path):
        return path.expanduser()
    return Path(str(path)).expanduser()


def safe_relative(path, roots = None):
    path = as_path(path)
    roots = list(roots or DEFAULT_SEARCH_ROOTS)
    for root in roots:
        root = as_path(root)
        try:
            return str(path.relative_to(root))
        except Exception:
            pass
    return str(path)


def finite_1d(values):
    arr = np.asarray(values, dtype=np.float64).ravel()
    return arr[np.isfinite(arr)]


def robust_limits(arr, low=2.0, high=98.0):
    values = finite_1d(arr)
    if values.size == 0:
        return None, None
    vmin = float(np.percentile(values, low))
    vmax = float(np.percentile(values, high))
    if np.isclose(vmin, vmax):
        vmin, vmax = float(np.min(values)), float(np.max(values))
    if np.isclose(vmin, vmax):
        delta = 1.0 if np.isclose(vmin, 0.0) else abs(vmin) * 0.01
        vmin -= delta
        vmax += delta
    return vmin, vmax


def show_dataframe(df, title = None):
    if title and Markdown is not None and display is not None:
        display(Markdown(f"### {title}"))
    if display is not None:
        display(df)
    else:
        print(("\n" + title) if title else "")
        print(df.to_string(index=False))


def flatten_metadata(mapping, prefix=""):
    rows = []
    if not isinstance(mapping, dict):
        return rows
    for key, value in mapping.items():
        full_key = f"{prefix}.{key}" if prefix else str(key)
        if isinstance(value, dict):
            rows.extend(flatten_metadata(value, prefix=full_key))
        elif isinstance(value, np.ndarray):
            rows.append((full_key, np.array2string(value, precision=4, threshold=8)))
        else:
            rows.append((full_key, value))
    return rows


def component_feature_key(name):
    text = str(name)
    if " | " in text:
        return text.split(" | ", 1)[1]
    return text


def component_channel(name):
    text = str(name)
    if " | " in text:
        return text.split(" | ", 1)[0]
    return ""



STAGE_ORIGINAL = "original_raw"
STAGE_PROCESSED = "processed_pre_normalized"
STAGE_NORMALIZED = "processed_normalized"

STAGE_LABELS = {
    STAGE_ORIGINAL: "Original/raw",
    STAGE_PROCESSED: "Processed/pre-normalized",
    STAGE_NORMALIZED: "Processed/normalized",
}

TABLE_SUFFIXES = {".csv", ".parquet", ".pq", ".xlsx", ".xls", ".feather"}
FEATURE_SUFFIXES = {".npy"}
VISUALIZABLE_SUFFIXES = TABLE_SUFFIXES | FEATURE_SUFFIXES


def normalize_stage(stage):
    """Accept internal stage keys or user-facing labels."""
    if stage is None:
        return None
    text = str(stage).strip().lower().replace("_", " ").replace("-", " ")
    if text in {"original", "raw", "original raw", "original/raw"}:
        return STAGE_ORIGINAL
    if text in {"processed", "pre normalized", "prenormalized", "processed pre normalized", "processed/pre normalized", "processed/pre-normalized"}:
        return STAGE_PROCESSED
    if text in {"normalized", "processed normalized", "processed/normalized"}:
        return STAGE_NORMALIZED
    if stage in STAGE_LABELS:
        return stage
    raise ValueError(f"Unknown stage {stage!r}. Use one of: {list(STAGE_LABELS.values())}")


def classify_visualizable_file(path):
    """Classify one file into the three visualization stages.

    Rules match the project pipeline:
    - table files are original/raw data
    - `.npy` under `Raw_Features` is processed/pre-normalized
    - `.npy` under `Normalized` is processed/normalized
    - other `.npy` feature files are treated as processed/pre-normalized
    """
    path = as_path(path)
    suffix = path.suffix.lower()
    parts = {part.lower() for part in path.parts}

    if suffix in TABLE_SUFFIXES:
        return STAGE_ORIGINAL
    if suffix == ".npy":
        if NORMALIZED_DIR_NAME.lower() in parts:
            return STAGE_NORMALIZED
        return STAGE_PROCESSED
    return None


def list_visualizable_files(
    roots = None,
    stage = None,
    include_npy = True,
    include_csv = True,
    include_table = True,
    max_files = 5000,
):
    """Discover visualizable files without requiring exact paths.

    When `stage` is given, only files from that stage are returned:
    - `Original/raw`: table files such as `.csv`
    - `Processed/pre-normalized`: `.npy` feature payloads before normalization
    - `Processed/normalized`: `.npy` feature payloads below `Normalized`
    """
    target_stage = normalize_stage(stage)
    roots = list(roots or DEFAULT_SEARCH_ROOTS)
    suffixes = set()
    if include_npy:
        suffixes.update(FEATURE_SUFFIXES)
    if include_csv:
        suffixes.add(".csv")
    if include_table:
        suffixes.update(TABLE_SUFFIXES)

    found = []
    seen = set()
    for root in roots:
        root = as_path(root)
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if len(found) >= max_files:
                break
            if not path.is_file():
                continue
            if path.suffix.lower() not in suffixes:
                continue
            detected_stage = classify_visualizable_file(path)
            if target_stage is not None and detected_stage != target_stage:
                continue
            key = str(path.resolve())
            if key in seen:
                continue
            found.append(path)
            seen.add(key)
    return sorted(found, key=lambda p: (classify_visualizable_file(p) or "", safe_relative(p, roots).lower(), natural_key(p)))


def list_visualizable_files_by_stage(
    roots = None,
    max_files = 5000,
):
    """Return discovered files grouped by stage."""
    roots = list(roots or DEFAULT_SEARCH_ROOTS)
    grouped = {key: [] for key in STAGE_LABELS}
    for stage in STAGE_LABELS:
        grouped[stage] = list_visualizable_files(roots=roots, stage=stage, max_files=max_files)
    return grouped


def print_file_inventory(roots = None, limit = 200):
    grouped = list_visualizable_files_by_stage(roots=roots, max_files=limit)
    total = sum(len(paths) for paths in grouped.values())
    print(f"Found {total} visualizable file(s).")
    flat = []
    for stage, paths in grouped.items():
        print(f"\n{STAGE_LABELS[stage]}: {len(paths)} file(s)")
        for i, path in enumerate(paths[:limit]):
            print(f"  [{i:04d}] {path}")
            flat.append(path)
        if len(paths) > limit:
            print(f"  ... truncated to first {limit} files for this stage.")
    return flat


In [ ]:
# =============================================================================
# LOADERS
# =============================================================================

def load_npy_payload(path):
    path = as_path(path)
    data = np.load(path, allow_pickle=True)
    if isinstance(data, np.lib.npyio.NpzFile):
        return {key: data[key] for key in data.files}
    if hasattr(data, "item"):
        try:
            data = data.item()
        except Exception:
            pass
    if not isinstance(data, dict):
        raise ValueError(f"{path} does not contain a dictionary payload.")
    return data


def is_feature_payload(payload):
    return isinstance(payload, dict) and "X" in payload and np.asarray(payload["X"]).ndim >= 2


def read_csv_guess(path, nrows = None):

    path = as_path(path)
    first = pd.read_csv(path, nrows=5)
    known_names = {"Time_s", "ACXXX_SN1 [g]", "ACXXX_SN2 [g]", "Tacho_Ch8 [Hz]"}
    has_known_header = bool(set(map(str, first.columns)).intersection(known_names))

    def numeric_like(value):
        try:
            float(str(value))
            return True
        except Exception:
            return False

    mostly_numeric_column_names = sum(numeric_like(c) for c in first.columns) >= max(1, int(0.6 * len(first.columns)))

    if has_known_header and not mostly_numeric_column_names:
        df = pd.read_csv(path, nrows=nrows)
    else:
        df = pd.read_csv(path, header=None, nrows=nrows)
        df.columns = [f"Sensor_{i + 1}" for i in range(df.shape[1])]

    return df


def read_table(path, nrows = None):
    path = as_path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return read_csv_guess(path, nrows=nrows)
    if suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
        return df.head(nrows) if nrows is not None else df
    if suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, nrows=nrows)
        return df
    if suffix == ".feather":
        df = pd.read_feather(path)
        return df.head(nrows) if nrows is not None else df
    raise ValueError(f"Unsupported table file type: {suffix}")


def numeric_columns(df):
    out = []
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().sum() > 0:
            out.append(col)
    return out


def infer_time_and_signal_columns(df, fs = None):
    columns = list(df.columns)
    time_col = None
    for candidate in ["Time_s", "time_s", "time", "Time", "t", "timestamp"]:
        if candidate in columns:
            time_col = candidate
            break

    num_cols = numeric_columns(df)
    if time_col is not None:
        signal_cols = [c for c in num_cols if c != time_col]
        t = pd.to_numeric(df[time_col], errors="coerce").to_numpy(dtype=np.float64)
    else:
        signal_cols = num_cols
        if fs is not None and fs > 0:
            t = np.arange(len(df), dtype=np.float64) / float(fs)
        else:
            t = np.arange(len(df), dtype=np.float64)

    signal_cols = [
        c for c in signal_cols
        if np.isfinite(pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=np.float64)).sum() > 0
    ]
    return t, signal_cols, time_col


In [ ]:
# =============================================================================
# FEATURE PAYLOAD VISUALIZATION: .npy from Raw_Features or Normalized
# =============================================================================

def payload_summary_tables(payload):
    meta_rows = flatten_metadata(payload.get("metadata", {}))
    label_rows = flatten_metadata(payload.get("labels", {}))
    if meta_rows:
        show_dataframe(pd.DataFrame(meta_rows, columns=["field", "value"]), "Metadata")
    if label_rows:
        show_dataframe(pd.DataFrame(label_rows, columns=["field", "value"]), "Labels")


def component_stats_table(payload):
    X = np.asarray(payload["X"])
    names = payload.get("component_names", [f"component_{i}" for i in range(X.shape[0])])
    rows = []
    for idx, name in enumerate(names):
        arr = np.asarray(X[idx])
        values = finite_1d(arr)
        if values.size == 0:
            rows.append({
                "index": idx,
                "component": name,
                "channel": component_channel(name),
                "feature_key": component_feature_key(name),
                "finite_count": 0,
                "nan_fraction": float(np.isnan(arr).mean()),
            })
            continue
        rows.append({
            "index": idx,
            "component": name,
            "channel": component_channel(name),
            "feature_key": component_feature_key(name),
            "shape": str(arr.shape),
            "finite_count": int(values.size),
            "nan_fraction": float(np.isnan(arr).mean()),
            "min": float(np.min(values)),
            "p01": float(np.percentile(values, 1)),
            "p05": float(np.percentile(values, 5)),
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
            "p95": float(np.percentile(values, 95)),
            "p99": float(np.percentile(values, 99)),
            "max": float(np.max(values)),
        })
    return pd.DataFrame(rows)


def select_component_indices(
    payload,
    component_filter = None,
    indices = None,
    max_components = None,
):
    X = np.asarray(payload["X"])
    names = payload.get("component_names", [f"component_{i}" for i in range(X.shape[0])])

    if indices is not None:
        selected = [int(i) for i in indices if 0 <= int(i) < len(names)]
    elif component_filter:
        pattern = re.compile(str(component_filter), flags=re.IGNORECASE)
        selected = [i for i, name in enumerate(names) if pattern.search(str(name))]
    else:
        preferred = []
        fallback = []
        for i, name in enumerate(names):
            key = component_feature_key(name)
            if key in {"amplitude_db", "cwt_amplitude_db", "speed_hz_map"}:
                preferred.append(i)
            else:
                fallback.append(i)
        selected = preferred + fallback

    return selected if max_components is None else selected[:int(max_components)]


def plot_feature_component(
    payload,
    component,
    robust = True,
    cmap = "viridis",
    frequency_limits = None,
    time_limits = None,
    figsize=(11, 5),
):
    X = np.asarray(payload["X"])
    names = payload.get("component_names", [f"component_{i}" for i in range(X.shape[0])])

    if isinstance(component, str):
        matches = [i for i, name in enumerate(names) if str(name) == component]
        if not matches:
            matches = [i for i, name in enumerate(names) if component.lower() in str(name).lower()]
        if not matches:
            raise ValueError(f"No component matched {component!r}.")
        idx = matches[0]
    else:
        idx = int(component)

    arr = np.asarray(X[idx], dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Component {idx} is not 2D. Shape: {arr.shape}")

    freq = np.asarray(payload.get("frequency_hz", np.arange(arr.shape[0])), dtype=np.float64)
    time = np.asarray(payload.get("time_s", np.arange(arr.shape[1])), dtype=np.float64)

    if len(freq) != arr.shape[0]:
        freq = np.arange(arr.shape[0], dtype=np.float64)
    if len(time) != arr.shape[1]:
        time = np.arange(arr.shape[1], dtype=np.float64)

    freq_mask = np.ones(len(freq), dtype=bool)
    time_mask = np.ones(len(time), dtype=bool)

    if frequency_limits is not None:
        lo, hi = frequency_limits
        freq_mask &= (freq >= lo) & (freq <= hi)
    if time_limits is not None:
        lo, hi = time_limits
        time_mask &= (time >= lo) & (time <= hi)

    arr_view = arr[np.ix_(freq_mask, time_mask)]
    freq_view = freq[freq_mask]
    time_view = time[time_mask]

    if robust:
        vmin, vmax = robust_limits(arr_view)
    else:
        values = finite_1d(arr_view)
        vmin, vmax = (float(np.min(values)), float(np.max(values))) if values.size else (None, None)

    fig, ax = plt.subplots(figsize=figsize)
    if len(time_view) > 1 and len(freq_view) > 1:
        extent = [float(time_view[0]), float(time_view[-1]), float(freq_view[0]), float(freq_view[-1])]
        im = ax.imshow(arr_view, aspect="auto", origin="lower", extent=extent, vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_xlabel("Time [s]" if "time_s" in payload else "Time index")
        ax.set_ylabel("Frequency [Hz]" if "frequency_hz" in payload else "Frequency index")
    else:
        im = ax.imshow(arr_view, aspect="auto", origin="lower", vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_xlabel("Time index")
        ax.set_ylabel("Frequency index")

    ax.set_title(f"{idx}: {names[idx]}")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


def plot_feature_profiles(payload, component, figsize=(11, 4)):
    X = np.asarray(payload["X"])
    names = payload.get("component_names", [f"component_{i}" for i in range(X.shape[0])])

    if isinstance(component, str):
        matches = [i for i, name in enumerate(names) if str(name) == component]
        if not matches:
            matches = [i for i, name in enumerate(names) if component.lower() in str(name).lower()]
        if not matches:
            raise ValueError(f"No component matched {component!r}.")
        idx = matches[0]
    else:
        idx = int(component)

    arr = np.asarray(X[idx], dtype=np.float32)
    freq = np.asarray(payload.get("frequency_hz", np.arange(arr.shape[0])), dtype=np.float64)
    time = np.asarray(payload.get("time_s", np.arange(arr.shape[1])), dtype=np.float64)
    if len(freq) != arr.shape[0]:
        freq = np.arange(arr.shape[0], dtype=np.float64)
    if len(time) != arr.shape[1]:
        time = np.arange(arr.shape[1], dtype=np.float64)

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(time, np.nanmean(arr, axis=0))
    ax.set_title(f"Mean over frequency vs time: {names[idx]}")
    ax.set_xlabel("Time [s]" if "time_s" in payload else "Time index")
    ax.set_ylabel("Mean value")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(freq, np.nanmean(arr, axis=1))
    ax.set_title(f"Mean over time vs frequency: {names[idx]}")
    ax.set_xlabel("Frequency [Hz]" if "frequency_hz" in payload else "Frequency index")
    ax.set_ylabel("Mean value")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()


def plot_feature_histograms(payload, selected_indices, bins=80, max_points_per_component=200_000):
    X = np.asarray(payload["X"])
    names = payload.get("component_names", [f"component_{i}" for i in range(X.shape[0])])

    for idx in selected_indices:
        values = finite_1d(X[idx])
        if values.size == 0:
            continue
        if values.size > max_points_per_component:
            rng = np.random.default_rng(42)
            values = values[rng.choice(values.size, size=max_points_per_component, replace=False)]
        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.hist(values, bins=bins)
        ax.set_title(f"Distribution: {idx}: {names[idx]}")
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")
        ax.grid(True, alpha=0.2)
        fig.tight_layout()
        plt.show()


def visualize_feature_npy(
    path,
    max_components = None,
    component_filter = None,
    indices = None,
    robust = True,
    cmap = "viridis",
    show_stats = True,
    show_profiles = True,
    show_histograms = False,
    frequency_limits = None,
    time_limits = None,
):
    path = as_path(path)
    payload = load_npy_payload(path)
    if not is_feature_payload(payload):
        raise ValueError(f"{path} is a dictionary .npy, but does not look like a feature payload.")

    X = np.asarray(payload["X"])
    names = payload.get("component_names", [f"component_{i}" for i in range(X.shape[0])])

    print(f"File: {path}")
    print(f"X shape: {X.shape}")
    print(f"Components: {len(names)}")
    payload_summary_tables(payload)

    stats = component_stats_table(payload)
    if show_stats:
        show_dataframe(stats, "Component statistics")

    selected = select_component_indices(
        payload,
        component_filter=component_filter,
        indices=indices,
        max_components=max_components,
    )
    if not selected:
        print("No components selected.")
        return payload, stats

    print("Selected components:")
    for idx in selected:
        print(f"  [{idx}] {names[idx]}")

    for idx in selected:
        plot_feature_component(
            payload,
            idx,
            robust=robust,
            cmap=cmap,
            frequency_limits=frequency_limits,
            time_limits=time_limits,
        )

    if show_profiles:
        for idx in selected:
            plot_feature_profiles(payload, idx)

    if show_histograms:
        plot_feature_histograms(payload, selected)

    return payload, stats


In [ ]:
# =============================================================================
# RAW CSV / TABLE VISUALIZATION
# =============================================================================

def table_stats(df):
    rows = []
    for col in numeric_columns(df):
        values = finite_1d(pd.to_numeric(df[col], errors="coerce"))
        if values.size == 0:
            continue
        rows.append({
            "column": col,
            "finite_count": int(values.size),
            "nan_fraction": float(pd.to_numeric(df[col], errors="coerce").isna().mean()),
            "min": float(np.min(values)),
            "p01": float(np.percentile(values, 1)),
            "p05": float(np.percentile(values, 5)),
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
            "p95": float(np.percentile(values, 95)),
            "p99": float(np.percentile(values, 99)),
            "max": float(np.max(values)),
        })
    return pd.DataFrame(rows)


def downsample_xy(x, y, max_points=50_000):
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) <= max_points:
        return x, y
    step = int(math.ceil(len(x) / max_points))
    return x[::step], y[::step]


def plot_raw_timeseries(df, fs = None, columns = None, max_columns = None, max_points=50_000):
    t, signal_cols, time_col = infer_time_and_signal_columns(df, fs=fs)
    if columns is None:
        columns = signal_cols if max_columns is None else signal_cols[:int(max_columns)]
    else:
        columns = [c for c in columns if c in df.columns]

    if not columns:
        print("No numeric signal columns available for time-series plotting.")
        return

    for col in columns:
        y = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
        x_plot, y_plot = downsample_xy(t, y, max_points=max_points)
        fig, ax = plt.subplots(figsize=(11, 3.5))
        ax.plot(x_plot, y_plot, linewidth=0.8)
        ax.set_title(f"Raw signal: {col}")
        ax.set_xlabel("Time [s]" if time_col or fs else "Sample index")
        ax.set_ylabel(str(col))
        ax.grid(True, alpha=0.25)
        fig.tight_layout()
        plt.show()


def infer_sampling_frequency(t, fs = None):
    if fs is not None and fs > 0:
        return float(fs)
    t = np.asarray(t, dtype=np.float64)
    finite = t[np.isfinite(t)]
    if finite.size < 3:
        return None
    diffs = np.diff(finite)
    diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
    if diffs.size == 0:
        return None
    median_dt = float(np.median(diffs))
    if median_dt <= 0:
        return None
    return 1.0 / median_dt


def plot_raw_fft(df, fs = None, columns = None, max_columns = None, max_points=262_144):
    t, signal_cols, time_col = infer_time_and_signal_columns(df, fs=fs)
    effective_fs = infer_sampling_frequency(t, fs=fs)
    if effective_fs is None:
        print("FFT skipped: sampling frequency cannot be inferred. Pass fs=... if this is a headerless file.")
        return

    if columns is None:
        columns = signal_cols if max_columns is None else signal_cols[:int(max_columns)]
    else:
        columns = [c for c in columns if c in df.columns]

    for col in columns:
        y = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
        y = y[np.isfinite(y)]
        if y.size < 4:
            continue
        if y.size > max_points:
            y = y[:max_points]
        y = y - np.mean(y)
        window = np.hanning(len(y))
        spectrum = np.abs(np.fft.rfft(y * window))
        freqs = np.fft.rfftfreq(len(y), d=1.0 / effective_fs)

        fig, ax = plt.subplots(figsize=(11, 3.5))
        ax.plot(freqs, spectrum, linewidth=0.8)
        ax.set_title(f"FFT magnitude: {col}")
        ax.set_xlabel("Frequency [Hz]")
        ax.set_ylabel("Magnitude")
        ax.grid(True, alpha=0.25)
        fig.tight_layout()
        plt.show()


def plot_raw_spectrogram(
    df,
    fs = None,
    column = None,
    nperseg = 4096,
    max_points = 500_000,
    cmap = "viridis",
):
    if spectrogram is None:
        print("Spectrogram skipped: scipy is not available.")
        return

    t, signal_cols, time_col = infer_time_and_signal_columns(df, fs=fs)
    effective_fs = infer_sampling_frequency(t, fs=fs)
    if effective_fs is None:
        print("Spectrogram skipped: sampling frequency cannot be inferred. Pass fs=... if this is a headerless file.")
        return

    if column is None:
        if not signal_cols:
            print("No signal columns available for spectrogram.")
            return
        column = signal_cols[0]
    if column not in df.columns:
        raise ValueError(f"Column {column!r} not found.")

    y = pd.to_numeric(df[column], errors="coerce").to_numpy(dtype=np.float64)
    y = y[np.isfinite(y)]
    if y.size < 4:
        print("Spectrogram skipped: not enough finite samples.")
        return
    if y.size > max_points:
        y = y[:max_points]
    y = y - np.mean(y)
    nperseg = min(int(nperseg), len(y))
    noverlap = min(int(0.75 * nperseg), max(0, nperseg - 1))
    f, tt, Sxx = spectrogram(y, fs=effective_fs, window="hann", nperseg=nperseg, noverlap=noverlap, scaling="spectrum", mode="magnitude")
    S_db = 20 * np.log10(Sxx + np.finfo(float).eps)
    vmin, vmax = robust_limits(S_db)

    fig, ax = plt.subplots(figsize=(11, 5))
    im = ax.imshow(S_db, aspect="auto", origin="lower", extent=[tt[0], tt[-1], f[0], f[-1]], cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(f"Quick spectrogram: {column}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Frequency [Hz]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Magnitude [dB]")
    fig.tight_layout()
    plt.show()




def infer_known_raw_sampling_frequency(path):
  
    text = str(as_path(path)).lower()
    if "machine_fault_data" in text or "imbalance" in text:
        return 50_000.0
    if "wibracje" in text or "wyważanie" in text or "wywazanie" in text:
        return 32_768.0
    return None

def visualize_table_file(
    path,
    nrows = 300_000,
    fs = None,
    columns = None,
    max_columns = None,
    show_fft = True,
    show_spectrogram = True,
):
    path = as_path(path)
    df = read_table(path, nrows=nrows)
    if fs is None:
        fs = infer_known_raw_sampling_frequency(path)

    print(f"File: {path}")
    print(f"Loaded shape: {df.shape}")
    show_dataframe(df.head(), "Preview")
    stats = table_stats(df)
    show_dataframe(stats, "Numeric column statistics")

    t, signal_cols, time_col = infer_time_and_signal_columns(df, fs=fs)
    print("Detected time column:", time_col)
    if fs is not None:
        print(f"Using sampling frequency: {fs:g} Hz")
    print("Detected numeric signal columns:", signal_cols[:20], "..." if len(signal_cols) > 20 else "")

    plot_raw_timeseries(df, fs=fs, columns=columns, max_columns=max_columns)
    if show_fft:
        plot_raw_fft(df, fs=fs, columns=columns, max_columns=max_columns)
    if show_spectrogram:
        spec_columns = columns if columns is not None else signal_cols
        for spec_column in spec_columns:
            plot_raw_spectrogram(df, fs=fs, column=spec_column)

    return df, stats


In [ ]:
# =============================================================================
# DISPATCH + COMPARISON
# =============================================================================

def visualize_file(
    path,
    *,
    nrows = 300_000,
    fs = None,
    max_components = None,
    component_filter = None,
    indices = None,
    columns = None,
    max_columns = None,
    robust = True,
    cmap = "viridis",
    show_stats = True,
    show_profiles = True,
    show_histograms = False,
    show_fft = True,
    show_spectrogram = True,
    frequency_limits = None,
    time_limits = None,
):

    path = as_path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()
    if suffix == ".npy":
        payload = load_npy_payload(path)
        if is_feature_payload(payload):
            return visualize_feature_npy(
                path,
                max_components=max_components,
                component_filter=component_filter,
                indices=indices,
                robust=robust,
                cmap=cmap,
                show_stats=show_stats,
                show_profiles=show_profiles,
                show_histograms=show_histograms,
                frequency_limits=frequency_limits,
                time_limits=time_limits,
            )
        else:
            print(f"{path} is a dictionary .npy, but not a feature payload. Keys:")
            print(sorted(payload.keys()))
            return payload

    if suffix in {".csv", ".parquet", ".pq", ".xlsx", ".xls", ".feather"}:
        return visualize_table_file(
            path,
            nrows=nrows,
            fs=fs,
            columns=columns,
            max_columns=max_columns,
            show_fft=show_fft,
            show_spectrogram=show_spectrogram,
        )

    raise ValueError(f"Unsupported file type: {suffix}")


def common_component_name(payload_a, payload_b, requested = None):
    names_a = [str(x) for x in payload_a.get("component_names", [])]
    names_b = [str(x) for x in payload_b.get("component_names", [])]
    if requested:
        exact_a = [x for x in names_a if x == requested]
        exact_b = [x for x in names_b if x == requested]
        if exact_a and exact_b:
            return requested
        candidates_a = [x for x in names_a if requested.lower() in x.lower()]
        candidates_b = [x for x in names_b if requested.lower() in x.lower()]
        for x in candidates_a:
            if x in candidates_b:
                return x
        raise ValueError(f"Component {requested!r} was not found in both payloads.")
    for x in names_a:
        if x in names_b:
            return x
    raise ValueError("No common component names found.")


def compare_feature_files(
    raw_feature_path,
    processed_feature_path,
    component = None,
    robust = True,
    cmap = "viridis",
):
    """Compare two feature `.npy` files, for example Raw_Features vs Normalized."""
    raw_payload = load_npy_payload(raw_feature_path)
    processed_payload = load_npy_payload(processed_feature_path)
    if not is_feature_payload(raw_payload) or not is_feature_payload(processed_payload):
        raise ValueError("Both files must be feature payload .npy files with an `X` array.")

    name = common_component_name(raw_payload, processed_payload, requested=component)
    raw_names = list(map(str, raw_payload["component_names"]))
    proc_names = list(map(str, processed_payload["component_names"]))
    raw_idx = raw_names.index(name)
    proc_idx = proc_names.index(name)

    raw_arr = np.asarray(raw_payload["X"][raw_idx], dtype=np.float32)
    proc_arr = np.asarray(processed_payload["X"][proc_idx], dtype=np.float32)

    raw_time = np.asarray(raw_payload.get("time_s", np.arange(raw_arr.shape[1])), dtype=np.float64)
    raw_freq = np.asarray(raw_payload.get("frequency_hz", np.arange(raw_arr.shape[0])), dtype=np.float64)
    proc_time = np.asarray(processed_payload.get("time_s", np.arange(proc_arr.shape[1])), dtype=np.float64)
    proc_freq = np.asarray(processed_payload.get("frequency_hz", np.arange(proc_arr.shape[0])), dtype=np.float64)

    raw_vmin, raw_vmax = robust_limits(raw_arr) if robust else (None, None)
    proc_vmin, proc_vmax = robust_limits(proc_arr) if robust else (None, None)

    fig, ax = plt.subplots(figsize=(11, 5))
    extent = [float(raw_time[0]), float(raw_time[-1]), float(raw_freq[0]), float(raw_freq[-1])]
    im = ax.imshow(raw_arr, aspect="auto", origin="lower", extent=extent, cmap=cmap, vmin=raw_vmin, vmax=raw_vmax)
    ax.set_title(f"Raw feature: {name}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Frequency [Hz]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 5))
    extent = [float(proc_time[0]), float(proc_time[-1]), float(proc_freq[0]), float(proc_freq[-1])]
    im = ax.imshow(proc_arr, aspect="auto", origin="lower", extent=extent, cmap=cmap, vmin=proc_vmin, vmax=proc_vmax)
    ax.set_title(f"Processed/normalized feature: {name}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Frequency [Hz]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()

    rows = []
    for label, arr in [("raw", raw_arr), ("processed", proc_arr)]:
        values = finite_1d(arr)
        rows.append({
            "file_role": label,
            "component": name,
            "shape": str(arr.shape),
            "min": float(np.min(values)) if values.size else np.nan,
            "p05": float(np.percentile(values, 5)) if values.size else np.nan,
            "mean": float(np.mean(values)) if values.size else np.nan,
            "std": float(np.std(values)) if values.size else np.nan,
            "p95": float(np.percentile(values, 95)) if values.size else np.nan,
            "max": float(np.max(values)) if values.size else np.nan,
        })
    stats = pd.DataFrame(rows)
    show_dataframe(stats, "Comparison statistics")
    return raw_payload, processed_payload, stats


In [ ]:
# =============================================================================
# WIDGET UI
# =============================================================================

def launch_file_visualizer(roots = None, max_files = 5000):

    if widgets is None or display is None:
        print("ipywidgets is not available. Use manual mode:")
        print('FILE_PATH = r"/path/to/file.npy"')
        print("visualize_file(FILE_PATH)")
        return None

    roots = list(roots or DEFAULT_SEARCH_ROOTS)
    files_by_stage = list_visualizable_files_by_stage(roots=roots, max_files=max_files)
    if not any(files_by_stage.values()):
        print("No visualizable files found. Use manual mode:")
        print('FILE_PATH = r"/path/to/file.npy"')
        print("visualize_file(FILE_PATH)")
        return None

    stage_options = [(label, key) for key, label in STAGE_LABELS.items()]
    stage_dd = widgets.Dropdown(options=stage_options, description="Type:", layout=widgets.Layout(width="50%"))
    file_dd = widgets.Dropdown(options=[], description="File:", layout=widgets.Layout(width="95%"))

    show_hist = widgets.Checkbox(value=False, description="Histograms")
    show_fft = widgets.Checkbox(value=True, description="FFT")
    show_spec = widgets.Checkbox(value=True, description="Spectrogram")
    run_button = widgets.Button(description="Visualize selected file", button_style="primary")
    output = widgets.Output()

    def make_file_options(stage_key):
        paths = files_by_stage.get(stage_key, [])
        if not paths:
            return [("No files found for this type", "")]
        return [
            (f"{i:04d} | {safe_relative(path, roots)}", str(path))
            for i, path in enumerate(paths)
        ]

    def update_file_dropdown(*_):
        stage_key = stage_dd.value
        file_dd.options = make_file_options(stage_key)
        if file_dd.options:
            file_dd.value = file_dd.options[0][1]

    def update_controls_visibility(*_):
        is_raw = stage_dd.value == STAGE_ORIGINAL
        show_fft.layout.display = "" if is_raw else "none"
        show_spec.layout.display = "" if is_raw else "none"

    def on_stage_change(change=None):
        update_file_dropdown()
        update_controls_visibility()
        with output:
            if clear_output is not None:
                clear_output(wait=True)
            stage_label = STAGE_LABELS[stage_dd.value]
            count = len(files_by_stage.get(stage_dd.value, []))
            print(f"{stage_label}: {count} file(s) available.")

    def _run(_=None):
        with output:
            if clear_output is not None:
                clear_output(wait=True)
            if not file_dd.value:
                print("No file selected for this type.")
                return
            is_raw = stage_dd.value == STAGE_ORIGINAL
            visualize_file(
                file_dd.value,
                max_components=None,
                component_filter=None,
                fs=None,  
                max_columns=None,
                show_histograms=bool(show_hist.value),
                show_fft=bool(show_fft.value) if is_raw else False,
                show_spectrogram=bool(show_spec.value) if is_raw else False,
            )

    stage_dd.observe(on_stage_change, names="value")
    run_button.on_click(_run)

    update_file_dropdown()
    update_controls_visibility()

    counts = widgets.HTML(
        value="<br>".join(
            f"<b>{STAGE_LABELS[key]}</b>: {len(paths)} file(s)"
            for key, paths in files_by_stage.items()
        )
    )

    controls = widgets.VBox([
        counts,
        stage_dd,
        file_dd,
        widgets.HBox([show_hist, show_fft, show_spec, run_button]),
    ])
    display(controls, output)
    return files_by_stage



In [ ]:
launch_file_visualizer()

Output()

{'original_raw': [WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/12.288.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/13.1072.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/14.336.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/15.1552.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/16.1792.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/17.2032.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/18.432.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/19.6608.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Machine_Fault_Data/imbalance/0g/20.2752.csv'),
  WindowsPath('c:/Users/szymo/Desktop/Wibracje/Datasety/Mach